# Ingest Observability Data into DOG with OpenTelemetry

**Module 2: Ingesting Observability Data** · [Lesson](course.md) · [Quiz](quiz02_ingesting_observability_data.ipynb)

In Module 1, the demo's telemetry reached Doris through pipelines that the course set up for you. In this Lab you build three pipelines of your own, one OpenTelemetry Collector for each signal, each configured by a YAML file next to this notebook: one reads the Doris FE log file, one pulls the metrics that Doris publishes, and one receives the spans that a Python program sends. You check in Doris that each signal arrived. Then you follow one of the failed orders from Module 1 into Doris: you find which tables hold its evidence, and which pipeline brought each piece.

Plan about 30 minutes. You need the DOG stack and the demo from Module 1, with the payment failure turned on (section 5 of Lab 1).

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## 1. Send logs to DOG

The first collector reads the Doris FE log, `fe.log`, and writes each log record to `otel_logs`. Its configuration is [otel_for_logs.yaml](otel_for_logs.yaml):

- The `filelog` receiver reads `/var/log/doris-fe/fe.log` from its first line. It joins the lines that belong to one record, such as a stack trace, and parses each record's time and severity with a regular expression.
- The `resource` processor names the source of the records: `service.name` is `doris-fe`. A log file does not say which service wrote it.
- The `batch` processor sends up to 10,000 records at a time, or what it has after 2 seconds.
- The `doris` exporter loads each batch into `otel_logs` with a Stream Load. It does not create tables (`create_schema: false`): the DOG Collector created them when Module 1 started it.
- The `file_storage` extension keeps the receiver's read position: how far it has read each file.

The cell asks Docker which host directory holds the FE's logs (the `doris-obs` container writes them to `.runtime/doris-logs/fe/`) and mounts that directory into the collector, read-only. Mounting the directory rather than the file matters: when the FE rotates its log, it renames `fe.log` and starts a new `fe.log`, and a collector that had only the old file mounted would never see the new lines. The cell also mounts `.runtime/collector-state/log2doris/` for the read position, and runs the collector as your user so that it can write there. Like the other two collectors, it joins the course's `doris-obs` Docker network and reaches Doris by its container name, `doris-obs` (see `endpoint` in the YAML files).

In [ ]:
lab.shell(r"""
set -euo pipefail

DORIS_CONTAINER="doris-obs"
CONFIG_DIR="level1/module02-data-ingest"
DORIS_FE_LOG_DIR="$(docker inspect --format '{{range .Mounts}}{{if eq .Destination "/opt/apache-doris/fe/log"}}{{.Source}}{{end}}{{end}}' "$DORIS_CONTAINER")"

if [ -z "$DORIS_FE_LOG_DIR" ]; then
  echo "Could not find the /opt/apache-doris/fe/log mount on the $DORIS_CONTAINER container."
  echo "Run module 1's prepare_environment() cell first, then rerun this cell."
  exit 1
fi

# Docker Desktop on macOS reports host mounts with a /host_mnt prefix.
DORIS_FE_LOG_DIR="${DORIS_FE_LOG_DIR#/host_mnt}"

if [ ! -f "$DORIS_FE_LOG_DIR/fe.log" ]; then
  echo "The Doris FE log was not found at $DORIS_FE_LOG_DIR/fe.log"
  exit 1
fi

# The collector saves how far it has read fe.log in this directory.
STATE_DIR="$PWD/.runtime/collector-state/log2doris"
mkdir -p "$STATE_DIR"

# Stop a running collector gracefully: it saves its read position and sends
# what it has read. Docker then removes it (--rm).
if docker container inspect doris-obs-log2doris >/dev/null 2>&1; then
  docker stop doris-obs-log2doris >/dev/null
  while docker container inspect doris-obs-log2doris >/dev/null 2>&1; do sleep 1; done
fi

# Run as your user, so that the collector can write its read position to STATE_DIR.
docker run --rm -d \
  --name doris-obs-log2doris \
  --network doris-obs \
  --user "$(id -u):$(id -g)" \
  -v "$PWD/$CONFIG_DIR/otel_for_logs.yaml:/etc/otel_for_logs.yaml:ro" \
  -v "$DORIS_FE_LOG_DIR:/var/log/doris-fe:ro" \
  -v "$STATE_DIR:/var/lib/otelcol/file_storage" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_logs.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-log2doris
""", title="Start the Doris log collector");

**Expected result**

The command log ends with `collector=/doris-obs-log2doris status=running`. When you rerun the cell, it first stops the running collector, which saves its read position and sends the records that it has read; the new collector continues from that position. Rerunning the cell therefore does not load the FE log into Doris a second time.

### Check that the FE logs reached Doris

A running collector does not show that any rows reached Doris. The next cell looks for them in `otel_logs`: the filelog receiver records the file name in the `log.file.name` attribute, the `regex_parser` operator in `otel_for_logs.yaml` fills `severity_text` from each line, and the `resource` processor names the source of every row: `service_name` is `doris-fe`. The batch processor and the load into Doris take a few seconds, so the cell reruns the query until rows arrive.

In [ ]:
fe_logs = lab.wait_for_rows(
    """
    SELECT service_name, severity_text, COUNT(*) AS log_rows, MAX(timestamp) AS latest_log
    FROM otel_logs
    WHERE log_attributes['log.file.name'] = 'fe.log'
    GROUP BY service_name, severity_text
    ORDER BY service_name, severity_text
    """,
    title="FE log rows in otel_logs",
    hint="Check that doris-obs-log2doris is running with docker ps, and read its log with docker logs doris-obs-log2doris.",
)

**Expected result**

One row per severity in the FE log, all with `service_name = doris-fe`: usually `INFO` and `WARN`, plus `ERROR` when the FE has logged errors. The collector starts at the oldest line of `fe.log` and exports up to 10,000 records per batch (`send_batch_size` in `otel_for_logs.yaml`), so right after it starts, `latest_log` can lag behind the current time. Rerun the cell a little later: `log_rows` has grown, and `latest_log` is within a few seconds of the current time in UTC, because the FE keeps writing to `fe.log` and the collector keeps loading the new lines.

### Where the table settings come from

`otel_for_logs.yaml` says nothing about partitions or how long Doris keeps the rows. An exporter applies such settings only when it creates a table, and `otel_logs` was created by the DOG Collector in Module 1, from its own configuration: `history_days: 7` and `create_history_days: 3`. They became the table's dynamic partitioning: one partition per day, created ahead of time, and dropped when it is more than 7 days old. List the settings of the telemetry tables:

In [ ]:
lab.sql(
    "SHOW DYNAMIC PARTITION TABLES FROM otel",
    title="Dynamic partitioning of the telemetry tables",
    columns=["TableName", "TimeUnit", "Start", "End", "Buckets", "ReplicationNum"],
);

**Expected result**

Eight rows, one for each table that the exporter created (tables that later modules create may follow), all with `TimeUnit` `DAY`, `Start` `-7`, and `End` `1`: Doris keeps the partitions of the last 7 days and creates the partition of the next day in advance. A setting such as `history_days: 10` in this Lab's YAML would change nothing, because the tables already exist; the exporter's `create_schema` decides whether it creates them at all. Module 5 looks at partitions in detail.

## 2. Send metrics to DOG

The second collector, [otel_for_metrics.yaml](otel_for_metrics.yaml), does not wait for data to arrive: its `prometheus` receiver pulls it. Every 15 seconds it requests `/metrics` from the FE (`doris-obs:8030`) and from the BE (`doris-obs:8040`), which publish their metrics as Prometheus text. The exporter writes each metric to the table for its type and uses `otel_metrics` as the prefix of the table names.

In [ ]:
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-metric2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-metric2doris \
  --network doris-obs \
  -v "$PWD/$CONFIG_DIR/otel_for_metrics.yaml:/etc/otel_for_metrics.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_metrics.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-metric2doris
""", title="Start the Doris metric collector");

**Expected result**

The command log ends with `collector=/doris-obs-metric2doris status=running`.

### Check that the metrics reached Doris

The Prometheus receiver scrapes the two targets in `otel_for_metrics.yaml` every 15 seconds and names each series after the scrape job (`service_name = 'doris_cluster'`) and the target (`service_instance_id`). The Doris exporter writes one table per metric type and uses the configured name `otel_metrics` as the prefix: gauges go to `otel_metrics_gauge`, counters to `otel_metrics_sum`, and so on. The next cell waits until both targets have rows in `otel_metrics_gauge`.

In [ ]:
doris_metrics = lab.wait_for_rows(
    """
    SELECT service_instance_id AS scrape_target,
           COUNT(DISTINCT metric_name) AS metric_names,
           COUNT(*) AS data_points,
           MAX(timestamp) AS latest_point
    FROM otel_metrics_gauge
    WHERE service_name = 'doris_cluster'
    GROUP BY service_instance_id
    ORDER BY scrape_target
    """,
    title="Doris metrics in otel_metrics_gauge",
    hint="Check that doris-obs-metric2doris is running with docker ps, and read its log with docker logs doris-obs-metric2doris.",
    min_rows=2,
)

**Expected result**

Two rows, usually within 30 seconds: `doris-obs:8030` (the FE, about 70 metric names) and `doris-obs:8040` (the BE, over 100). `latest_point` is close to the current time in UTC, and `data_points` grows with every scrape when you rerun the cell.

The collector's log (`docker logs doris-obs-metric2doris`) also shows a few `warn` lines that name `doris_be_tablet_version_num_distribution`: the BE publishes that metric as a histogram without bucket bounds, so the receiver drops it. The other metrics load normally, and each Stream Load response in the log reports `"NumberFilteredRows": 0`.

## 3. Send traces to DOG

This collector receives spans over OTLP, adds the `deployment.environment.name` resource attribute (the `resource` processor in `otel_for_traces.yaml`), and writes the spans to `otel_traces`. Unlike the other two collectors, it publishes its OTLP/HTTP receiver on the host, at `127.0.0.1:24328` (or the port in `DORIS_OBS_TRACES_OTLP_HTTP_PORT`), so that the Python code in the next section can send spans to it. The DOG stack's own Collector listens on `127.0.0.1:24318`; spans sent there never pass through this collector or its configuration.

The first line of the cell checks that no other program uses the host port.

In [ ]:
lab.check_trace_collector_port()
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-traces2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-traces2doris \
  --network doris-obs \
  -p "127.0.0.1:${DORIS_OBS_TRACES_OTLP_HTTP_PORT:-24328}:4318" \
  -v "$PWD/$CONFIG_DIR/otel_for_traces.yaml:/etc/otel_for_traces.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_traces.yaml

# Wait until the collector's log says that it is ready to receive data.
for attempt in $(seq 1 30); do
  docker logs doris-obs-traces2doris 2>&1 | grep "Everything is ready" >/dev/null && break
  sleep 1
done
docker logs doris-obs-traces2doris 2>&1 | grep -o "Everything is ready.*processing data."

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-traces2doris
docker port doris-obs-traces2doris
""", title="Start the trace collector");

**Expected result**

The port check shows `24328` as `available`, or as `used by this course` when you rerun the cell. The collector opens its receiver once its whole pipeline has started, so the cell waits for its log line `Everything is ready. Begin running and processing data.` The command log ends with that line, `collector=/doris-obs-traces2doris status=running`, and `4318/tcp -> 127.0.0.1:24328`.

## 4. Send a span and find it in Doris

The following code creates one span with the OpenTelemetry SDK, which the course's `requirements.txt` installs, and exports it to the trace collector over OTLP/HTTP.

In [ ]:
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

resource = Resource.create({"service.name": "python-demo-service"})

provider = TracerProvider(resource=resource)
# The trace collector's OTLP/HTTP receiver: http://127.0.0.1:24328 unless DORIS_OBS_TRACES_OTLP_HTTP_PORT is set.
processor = BatchSpanProcessor(OTLPSpanExporter(endpoint=f"{lab.trace_collector_url}/v1/traces"))
provider.add_span_processor(processor)
# Use this provider directly. The global trace.set_tracer_provider() accepts only one
# provider per Python process, so a rerun of this cell would keep using the first one.
tracer = provider.get_tracer("demo-tracer")

with tracer.start_as_current_span("demo-span") as span:
    span.set_attribute("custom.attribute", "hello-otel")
    span.set_attribute("http.status_code", 200)

# BatchSpanProcessor exports in the background. Export the queued span now.
provider.force_flush()
trace_id = format(span.get_span_context().trace_id, "032x")
print(f"Flushed the span. Trace ID: {trace_id}")


**Expected result**

The cell prints the 32-character trace ID of the span, for example `Flushed the span. Trace ID: 4bf92f3577b34da6a3ce929d0e0e4736`; yours differs on every run. `force_flush()` waits for the export attempt but does not report its outcome: when the collector is unreachable, the exporter only logs warnings such as `Failed to export spans batch` under the cell. Neither the print nor the flush shows that Doris stored the span, so the next cell looks for it.

### Find the span in Doris

Look the span up by its trace ID. The `environment` column reads the resource attribute that the trace collector adds. The Python code does not set it, so `course-lab` in that column shows that the span went through `doris-obs-traces2doris`.

In [ ]:
span_rows = lab.wait_for_rows(
    f"""
    SELECT trace_id, service_name, span_name, timestamp,
           resource_attributes['deployment.environment.name'] AS environment,
           span_attributes['custom.attribute'] AS custom_attribute
    FROM otel_traces
    WHERE trace_id = '{trace_id}'
    """,
    title="The span in otel_traces",
    hint="Check that doris-obs-traces2doris is running with docker ps, read its log with docker logs "
         "doris-obs-traces2doris, and look for export warnings under the previous cell.",
)

**Expected result**

One row, usually within 20 seconds: your trace ID, `python-demo-service`, `demo-span`, a `timestamp` in UTC, `environment = course-lab`, and `custom_attribute = hello-otel`. The delay comes from the collector's batch processor (`timeout: 10s` in `otel_for_traces.yaml`) and from Doris committing the load.

### Try it: change the collector configuration

The collector reads `otel_for_traces.yaml` only when it starts.

1. In `otel_for_traces.yaml`, change `value: course-lab` to another value, such as `value: my-laptop`, and save the file.
2. Rerun the cell in section 3 to start the collector again with the new file. If the cell fails, the collector could not start with the edited file: check the change, or undo it, and rerun the cell.
3. Rerun the two code cells in this section. The new span shows your value in `environment`.

Your chosen value also satisfies the completion check below. To restore the original example instead, put `value: course-lab` back in `otel_for_traces.yaml`, save it, rerun section 3 to restart the collector, and send a new span with the two code cells of section 4. Existing spans keep the value they were sent with.

## 5. Follow one failed order into Doris

Since you turned on the payment failure in Module 1, about half of the demo's orders fail. The evidence of a failed order is spread over three tables. It got there through pipelines like the ones you just built: the demo's services send their spans, logs, and metrics over OTLP to the demo's own Collector, which forwards them to the DOG Collector, whose `doris` exporter loads them into Doris.

Find the most recent failed order. The checkout service records each order as a span named `oteldemo.CheckoutService/PlaceOrder`, and the span's status is `STATUS_CODE_ERROR` when the order failed:

In [ ]:
failed_order = lab.wait_for_rows(
    """
    SELECT trace_id, timestamp
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 30 MINUTE
    ORDER BY timestamp DESC
    LIMIT 1
    """,
    title="The most recent failed order",
    hint="No order failed in the last 30 minutes. Turn on paymentFailure in section 5 of Lab 1, "
         "wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)
failed_trace = failed_order["trace_id"][0]

**Expected result**

One row: the trace ID of the order and the time of its checkout span, in UTC. The next cells look this trace ID up in each table.

### The spans in `otel_traces`

Every span of the order carries its trace ID. Count the spans by service, and how many of them have an error status:

In [ ]:
lab.sql(f"""
SELECT service_name,
       COUNT(*) AS spans,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS error_spans
FROM otel_traces
WHERE trace_id = '{failed_trace}'
GROUP BY service_name
ORDER BY error_spans DESC, spans DESC
""", title="The spans of the failed order, by service");

**Expected result**

About ten services and 25 to 40 spans; the number depends on how many items the order had. Five services have error spans, listed first: `frontend` with three, `checkout` and `frontend-proxy` with two, and `payment` and `load-generator` with one. They are the services on the order's path, from the load generator through the storefront's proxy and the frontend to checkout and payment. The other services, such as `currency`, `product-catalog`, `quote`, `cart`, and `shipping`, did their part without an error. Each service's OpenTelemetry SDK sent its own spans, and the shared trace ID joins them into one request. Module 3 opens this trace in Grafana and finds the span where the error started.

### The logs in `otel_logs`

An OpenTelemetry SDK stamps each log record with the trace ID of the request that the service was handling, so the same ID finds the order's logs:

In [ ]:
lab.sql(f"""
SELECT timestamp, service_name, severity_number, severity_text, LEFT(body, 90) AS body_start
FROM otel_logs
WHERE trace_id = '{failed_trace}'
ORDER BY timestamp
""", title="The logs of the failed order");

**Expected result**

A few rows. The first is the storefront proxy's access log line: `service_name` is `frontend-proxy`, and its body has `"POST /api/checkout HTTP/1.1" 500`. The others are `INFO` records of services that did their part, such as `cart`, `currency`, and `quote`. No row comes from `payment` or `checkout`: in this demo, those two services write their logs only to their containers' output (`docker logs doris-obs-demo-payment` shows the payment error) and send no logs to a Collector. The proxy's line has `severity_number` 0 and an empty `severity_text`, because an access log line has no severity. A search for error logs therefore misses this failure; Module 3 searches the access log instead.

### The metrics in `otel_metrics_sum`

The demo's Collector also turns spans into metrics: its `spanmetrics` connector counts the spans of each service, span name, and status, and sends the counts as the metric `traces.span.metrics.calls`. A count only grows, so it is a sum and lands in `otel_metrics_sum`; each data point is the total since the connector started counting (`start_time`). Read the latest counts of the payment service's charge requests:

In [ ]:
lab.sql("""
SELECT CAST(attributes['status.code'] AS STRING) AS status_code,
       MAX_BY(value, timestamp) AS charges_counted,
       MAX(start_time) AS counting_since,
       MAX(timestamp) AS latest_point
FROM otel_metrics_sum
WHERE metric_name = 'traces.span.metrics.calls'
  AND service_name = 'payment'
  AND attributes['span.name'] = 'grpc.oteldemo.PaymentService/Charge'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
GROUP BY status_code
ORDER BY status_code
""", title="Charge requests counted by the demo Collector, by status");

**Expected result**

Two rows: `STATUS_CODE_ERROR`, the charge requests that failed, and `STATUS_CODE_UNSET`, those that succeeded, both counted since `counting_since`, when the demo started. The failures began when you turned on the flag. A counter answers "how many" and "how often" cheaply, at any time scale, but none of its points leads to an order: only the trace has the order's spans.

| Table | Evidence of the failed order | How it arrived |
| --- | --- | --- |
| `otel_traces` | The order's spans, with an error status on `payment` and the services above it | Each service's SDK, then the demo Collector and the DOG Collector |
| `otel_logs` | The proxy's access log line with HTTP 500, and `INFO` records of other services; nothing from `payment` or `checkout` | The same path |
| `otel_metrics_sum` | Counts of failed and successful charge requests, not the order itself | Made from the spans by the demo Collector's `spanmetrics` connector |

## 6. Exercise: count the failed checkouts in the access log

Every request through the storefront's proxy leaves one access log line in `otel_logs`, with the HTTP status after the request, for example `"POST /api/checkout HTTP/1.1" 500`. Orders placed in the storefront page carry a query string after the path, as in `"POST /api/checkout?currencyCode=USD HTTP/1.1" 500`. Write a query that counts the checkout requests of the last 15 minutes by HTTP status. Tip: `REGEXP_EXTRACT(body, '<pattern>', 1)` returns the part of `body` that the first group in the pattern matches.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.sql("""
SELECT REGEXP_EXTRACT(body, '"POST /api/checkout[^"]*" ([0-9]+)', 1) AS http_status,
       COUNT(*) AS requests
FROM otel_logs
WHERE service_name = 'frontend-proxy'
  AND body LIKE '%"POST /api/checkout%'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
GROUP BY http_status
ORDER BY http_status
""", title="Checkout requests of the last 15 minutes, by HTTP status");

**Expected result**

Two rows: `200` for the orders that went through and `500` for those that failed. Since you turned on the flag, about half of the requests are `500`; with so few orders, the share can be well above or below half. Requests from before the flag are all `200`. Each `500` matches a checkout span with an error status, so the access log and the traces count the same failures.

## You are done when

- `otel_logs` has the FE log with `service_name = doris-fe`, and rerunning section 1 does not load it again;
- `otel_metrics_gauge` has data points from both of the Doris scrape targets;
- your span is in `otel_traces` with the `environment` value you chose (`course-lab` before the Try it);
- you have found the spans, the logs, and the metric counts of one failed order.

The three collectors keep running and use little memory; the later modules do not need them, and `lab.teardown()` removes them with the rest of the course. To stop them sooner, run `docker stop doris-obs-log2doris doris-obs-metric2doris doris-obs-traces2doris` in a terminal.

## Next

Open [Quiz 2: Ingesting Observability Data](quiz02_ingesting_observability_data.ipynb), then continue to [Module 3: Visualizing Observability Data](../module03-data-visualization/course.md).